# SciELO relational model: upload of the dump (2026)

In [1]:
import json
import zipfile
import pandas as pd
import xmltodict
import google.auth
from google.cloud import bigquery
from google.cloud.bigquery.job import LoadJobConfig, WriteDisposition, CreateDisposition

In [ ]:
creds, _ = google.auth.default()
bq_client = bigquery.Client(project='multiobs', credentials=creds.with_quota_project(None))

In [ ]:
bq_client.query('CREATE SCHEMA IF NOT EXISTS multiobs.publicdb_scielo_2026_09').result()

In [ ]:
def save_to_bqtable(ts, Ti, disposition):

    table_id = 'multiobs.publicdb_scielo_2026_09.'+ts
    load_config = LoadJobConfig(create_disposition=CreateDisposition.CREATE_IF_NEEDED,
                               write_disposition=disposition)

    job = bq_client.load_table_from_dataframe(
        dataframe=Ti, destination=table_id,job_config=load_config
    )
    job.result()

In [ ]:
z = zipfile.ZipFile('data/scielo_2026/scielo_articles.zip')
names = [n for n in z.namelist() if n.endswith('.xml')]
print(len(names), 'xml files')

disposition = WriteDisposition.WRITE_TRUNCATE
for i in range(0, len(names), 10000):
    works = [json.dumps(xmltodict.parse(z.read(n))) for n in names[i:i+10000]]
    save_to_bqtable('works_json', pd.DataFrame({'works': works}), disposition)
    disposition = WriteDisposition.WRITE_APPEND
    print(i + len(works), 'works uploaded')